
# 11 — REINFORCE, Vanilla, Advantage, TD-Residual, and Q-Weighted Policy Gradient Methods

This notebook compares five policy-gradient update methods using the same three trajectories.

At every decision state, four actions are available:

$$
\mathcal{A}=\{\mathrm{Up},\mathrm{Down},\mathrm{Left},\mathrm{Right}\}.
$$

The common policy-gradient term is

$$
\nabla_{\theta}\log \pi_{\theta}(A_t\mid S_t).
$$

The methods differ mainly in the learning signal that multiplies this term.


## 1. Four-action softmax policy

At every decision state, the action order in this notebook is

$$
[\mathrm{Up},\mathrm{Down},\mathrm{Left},\mathrm{Right}].
$$

For state $s$, the policy logits are

$$
\mathbf z(s)=
\begin{bmatrix}
z_{\mathrm{Up}}\\
z_{\mathrm{Down}}\\
z_{\mathrm{Left}}\\
z_{\mathrm{Right}}
\end{bmatrix}.
$$

Softmax converts the logits into action probabilities:

$$
\pi_{\theta}(a_j\mid s)
=
\frac{e^{z_j(s)}}{\sum_k e^{z_k(s)}}.
$$

Initially,

$$
\mathbf z(s)=
\begin{bmatrix}
0\\
0\\
0\\
0
\end{bmatrix}
$$

so

$$
\boldsymbol{\pi}(s)=
\begin{bmatrix}
0.25\\
0.25\\
0.25\\
0.25
\end{bmatrix}.
$$

For softmax, the gradient with respect to the logits is

$$
\boxed{
\nabla_{\mathbf z}
\log \pi_{\theta}(A_t\mid S_t)
=
\mathbf e_{A_t}
-
\boldsymbol{\pi}(S_t)
}
$$

where $\mathbf e_{A_t}$ is the one-hot vector of the action actually selected.

For the action order used here,

$$
\mathbf e_{\mathrm{Up}}
=
\begin{bmatrix}
1\\0\\0\\0
\end{bmatrix},
\qquad
\mathbf e_{\mathrm{Down}}
=
\begin{bmatrix}
0\\1\\0\\0
\end{bmatrix},
$$

$$
\mathbf e_{\mathrm{Left}}
=
\begin{bmatrix}
0\\0\\1\\0
\end{bmatrix},
\qquad
\mathbf e_{\mathrm{Right}}
=
\begin{bmatrix}
0\\0\\0\\1
\end{bmatrix}.
$$

### Example: Right is selected initially

Because Right was selected,

$$
\mathbf e_{A_t}
=
\mathbf e_{\mathrm{Right}}
=
\begin{bmatrix}
0\\
0\\
0\\
1
\end{bmatrix}.
$$

The current policy is

$$
\boldsymbol{\pi}(S_t)
=
\begin{bmatrix}
0.25\\
0.25\\
0.25\\
0.25
\end{bmatrix}.
$$

Therefore,

$$
\begin{aligned}
\nabla_{\mathbf z}
\log \pi_{\theta}(\mathrm{Right}\mid S_t)
&=
\mathbf e_{\mathrm{Right}}
-
\boldsymbol{\pi}(S_t)\\[4pt]
&=
\begin{bmatrix}
0\\
0\\
0\\
1
\end{bmatrix}
-
\begin{bmatrix}
0.25\\
0.25\\
0.25\\
0.25
\end{bmatrix}\\[6pt]
&=
\begin{bmatrix}
-0.25\\
-0.25\\
-0.25\\
0.75
\end{bmatrix}.
\end{aligned}
$$

So the selected action Right receives a positive gradient component $+0.75$, while the other three actions receive $-0.25$ each.


## 1.1 How θ, z, softmax, and the action one-hot vector are connected

The general policy-gradient equation is written with the policy parameters $\theta$:

$$
\pi_{\theta}(a\mid s).
$$

**How to read it:** $\pi_{\theta}(a\mid s)$ is the probability that the current policy chooses action $a$ when it is in state $s$. The symbol $\theta$ represents the trainable parameters of the policy.

For a neural-network policy, the computation is

$$
S_t
\;\longrightarrow\;
\text{policy network with parameters }\theta
\;\longrightarrow\;
\mathbf z(S_t;\theta)
\;\longrightarrow\;
\operatorname{softmax}
\;\longrightarrow\;
\boldsymbol{\pi}_{\theta}(S_t).
$$

Here,

- $S_t$: state observed at time $t$;
- $\theta$: trainable network weights and biases;
- $\mathbf z(S_t;\theta)$: vector of network outputs before softmax, called **logits**;
- $\boldsymbol{\pi}_{\theta}(S_t)$: vector of action probabilities after softmax.

For four actions,

$$
\mathbf z(S_t;\theta)
=
\begin{bmatrix}
z_{\mathrm{Up}}\\
z_{\mathrm{Down}}\\
z_{\mathrm{Left}}\\
z_{\mathrm{Right}}
\end{bmatrix},
\qquad
\boldsymbol{\pi}_{\theta}(S_t)
=
\begin{bmatrix}
P(\mathrm{Up}\mid S_t)\\
P(\mathrm{Down}\mid S_t)\\
P(\mathrm{Left}\mid S_t)\\
P(\mathrm{Right}\mid S_t)
\end{bmatrix}.
$$

**How to read it:** the first vector contains four unrestricted scores; softmax converts those scores into four probabilities that sum to $1$.

### What is $\mathbf e_{A_t}$?

$\mathbf e_{A_t}$ is the **one-hot vector** for the action actually selected at time $t$.

With action order

$$
[\mathrm{Up},\mathrm{Down},\mathrm{Left},\mathrm{Right}],
$$

the one-hot vectors are

$$
\mathbf e_{\mathrm{Up}}
=
\begin{bmatrix}
1\\0\\0\\0
\end{bmatrix},
\qquad
\mathbf e_{\mathrm{Down}}
=
\begin{bmatrix}
0\\1\\0\\0
\end{bmatrix},
$$

$$
\mathbf e_{\mathrm{Left}}
=
\begin{bmatrix}
0\\0\\1\\0
\end{bmatrix},
\qquad
\mathbf e_{\mathrm{Right}}
=
\begin{bmatrix}
0\\0\\0\\1
\end{bmatrix}.
$$

For softmax,

$$
\nabla_{\mathbf z}
\log \pi_{\theta}(A_t\mid S_t)
=
\mathbf e_{A_t}
-
\boldsymbol{\pi}_{\theta}(S_t).
$$

**How to read it:** take the one-hot vector of the action that was actually selected and subtract the current probability vector. This gives the gradient with respect to the four logits.

For example, if Right is selected while all actions have probability $0.25$,

$$
\nabla_{\mathbf z}
\log \pi_{\theta}(\mathrm{Right}\mid S_t)
=
\begin{bmatrix}
0\\0\\0\\1
\end{bmatrix}
-
\begin{bmatrix}
0.25\\0.25\\0.25\\0.25
\end{bmatrix}
=
\begin{bmatrix}
-0.25\\-0.25\\-0.25\\0.75
\end{bmatrix}.
$$

### Why does the general equation use $\nabla_{\theta}$, but the manual calculation uses $\nabla_{\mathbf z}$?

For a neural network, the complete gradient is obtained by the chain rule:

$$
\nabla_{\theta}
\log \pi_{\theta}(A_t\mid S_t)
=
\left(
\frac{\partial \mathbf z(S_t;\theta)}
{\partial \theta}
\right)^{T}
\nabla_{\mathbf z}
\log \pi_{\theta}(A_t\mid S_t).
$$

**How to read it:** first calculate how the log-probability changes with the output logits $z$, then backpropagate that change through the network to the actual parameters $\theta$.

For the manual calculations in this notebook, we deliberately use a simplified parameterization:

$$
\boxed{\theta_s=\mathbf z(s)}.
$$

Therefore, in these manual examples,

$$
\boxed{
\nabla_{\theta_s}\log\pi
=
\nabla_{\mathbf z}\log\pi
}.
$$

This is why the numerical examples can update the logits directly. In a real neural-network implementation, backpropagation performs the extra chain-rule step automatically.


## 1.2 How to read a policy-gradient expectation

A general policy-gradient expression is

$$
\nabla_{\theta}J(\theta)
=
\mathbb E_{\tau\sim\pi_{\theta}}
\left[
\sum_{t=0}^{T-1}
W_t
\nabla_{\theta}
\log \pi_{\theta}(A_t\mid S_t)
\right].
$$

### Meaning of every symbol

- $J(\theta)$: objective function; the expected performance of the current policy.
- $\theta$: trainable parameters of the policy.
- $\nabla_{\theta}J(\theta)$: direction in parameter space that increases the objective.
- $\tau$: one complete trajectory or episode.
- $\pi_{\theta}$: the current policy.
- $\tau\sim\pi_{\theta}$: the trajectory is generated by following the current policy.
- $\mathbb E_{\tau\sim\pi_{\theta}}[\cdot]$: expectation over trajectories that could be generated by the current policy.
- $T$: number of decision steps in a trajectory.
- $t$: time-step index inside one trajectory.
- $S_t$: state at step $t$.
- $A_t$: action selected at step $t$.
- $W_t$: learning signal used by the particular policy-gradient method.
- $\pi_{\theta}(A_t\mid S_t)$: probability assigned to the selected action.
- $\log \pi_{\theta}(A_t\mid S_t)$: log-probability of the selected action.
- $\nabla_{\theta}\log \pi_{\theta}(A_t\mid S_t)$: gradient of the selected action's log-probability with respect to the policy parameters.

### How to read the complete equation

First look inside the summation:

$$
W_t
\nabla_{\theta}
\log \pi_{\theta}(A_t\mid S_t).
$$

This is the contribution from one decision step.

Then

$$
\sum_{t=0}^{T-1}
W_t
\nabla_{\theta}
\log \pi_{\theta}(A_t\mid S_t)
$$

means:

> Add the gradient contributions from all decision steps inside one trajectory.

Finally,

$$
\mathbb E_{\tau\sim\pi_{\theta}}[\cdot]
$$

means:

> Average that trajectory-level quantity over trajectories generated by the same current policy.

The expectation is **not over different policies**. There is one current policy $\pi_{\theta}$, but that policy can generate many different trajectories because action selection and/or the environment can be stochastic.

For a finite batch of $N$ sampled trajectories,

$$
\mathbb E_{\tau\sim\pi_{\theta}}[f(\tau)]
\approx
\frac{1}{N}
\sum_{i=1}^{N}
f(\tau_i).
$$

Therefore,

$$
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{t=0}^{T_i-1}
(\cdots)
$$

is read as:

> For each trajectory, first sum over its time steps; then average those trajectory results over the $N$ sampled trajectories.

## 2. Three trajectories

### Trajectory 1: 3 decisions

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Trajectory 2: 4 decisions

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Trajectory 3: 5 decisions

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$

## 3. Monte Carlo return calculations

Use

$$
G_t=R_{t+1}+\gamma G_{t+1},\qquad \gamma=0.9.
$$

Every return is shown explicitly.

### Trajectory 1

$$
G_2=R_{3}=3=3.0000
$$


$$
\begin{aligned}G_1 &= R_{2} + \gamma R_{3}\\&= 1 + 0.9(3)\\&= 3.7000\end{aligned}
$$


$$
\begin{aligned}G_0 &= R_{1} + \gamma R_{2} + \gamma^{2}R_{3}\\&= 2 + 0.9(1) + 0.9^{2}(3)\\&= 5.3300\end{aligned}
$$


### Trajectory 2

$$
G_3=R_{4}=2=2.0000
$$


$$
\begin{aligned}G_2 &= R_{3} + \gamma R_{4}\\&= -1 + 0.9(2)\\&= 0.8000\end{aligned}
$$


$$
\begin{aligned}G_1 &= R_{2} + \gamma R_{3} + \gamma^{2}R_{4}\\&= 1 + 0.9(-1) + 0.9^{2}(2)\\&= 1.7200\end{aligned}
$$


$$
\begin{aligned}G_0 &= R_{1} + \gamma R_{2} + \gamma^{2}R_{3} + \gamma^{3}R_{4}\\&= -2 + 0.9(1) + 0.9^{2}(-1) + 0.9^{3}(2)\\&= -0.4520\end{aligned}
$$


### Trajectory 3

$$
G_4=R_{5}=4=4.0000
$$


$$
\begin{aligned}G_3 &= R_{4} + \gamma R_{5}\\&= 1 + 0.9(4)\\&= 4.6000\end{aligned}
$$


$$
\begin{aligned}G_2 &= R_{3} + \gamma R_{4} + \gamma^{2}R_{5}\\&= 2 + 0.9(1) + 0.9^{2}(4)\\&= 6.1400\end{aligned}
$$


$$
\begin{aligned}G_1 &= R_{2} + \gamma R_{3} + \gamma^{2}R_{4} + \gamma^{3}R_{5}\\&= -2 + 0.9(2) + 0.9^{2}(1) + 0.9^{3}(4)\\&= 3.5260\end{aligned}
$$


$$
\begin{aligned}G_0 &= R_{1} + \gamma R_{2} + \gamma^{2}R_{3} + \gamma^{3}R_{4} + \gamma^{4}R_{5}\\&= 1 + 0.9(-2) + 0.9^{2}(2) + 0.9^{3}(1) + 0.9^{4}(4)\\&= 4.1734\end{aligned}
$$


### How to read the return equation

The Monte Carlo return is

$$
G_t
=
R_{t+1}
+
\gamma R_{t+2}
+
\gamma^2R_{t+3}
+\cdots.
$$

Equivalent recursive form:

$$
G_t
=
R_{t+1}
+
\gamma G_{t+1}.
$$

Meaning:

- $G_t$: discounted return from time $t$ onward;
- $R_{t+1}$: reward received immediately after taking action $A_t$;
- $\gamma$: discount factor, $0\le\gamma\le1$;
- $G_{t+1}$: return beginning at the next time step.

**How to read it:** $G_t$ contains the actual rewards observed after the current action until the episode ends, with later rewards discounted more strongly.

There is **no expectation in this equation**. Once one trajectory has been sampled and completed, its rewards are known, so $G_t$ is calculated directly from that trajectory.

In [7]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

ACTIONS = ["Up", "Down", "Left", "Right"]
ACTION_TO_INDEX = {a:i for i,a in enumerate(ACTIONS)}
alpha = 0.10
gamma = 0.90

trajectories = [
    [("S0","Right", 2.0), ("S1","Up", 1.0), ("S2","Right", 3.0)],
    [("S0","Left", -2.0), ("S1","Right", 1.0), ("S2","Down", -1.0), ("S3","Right", 2.0)],
    [("S0","Right", 1.0), ("S1","Left", -2.0), ("S2","Up", 2.0), ("S3","Down", 1.0), ("S4","Right", 4.0)]
]

def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - np.max(z))
    return e / e.sum()

def grad_log_softmax(p, action):
    g = -np.asarray(p, dtype=float).copy()
    g[ACTION_TO_INDEX[action]] += 1.0
    return g

def discounted_returns(traj):
    G = 0.0
    vals = []
    for s,a,r in reversed(traj):
        G = r + gamma*G
        vals.append(G)
    return list(reversed(vals))

rows=[]
for episode,traj in enumerate(trajectories,1):
    Gs=discounted_returns(traj)
    for t,((s,a,r),G) in enumerate(zip(traj,Gs)):
        rows.append({"Trajectory":episode,"t":t,"State":s,"Action":a,"Reward":r,"G_t":G})
pd.DataFrame(rows)


,Trajectory,t,State,Action,Reward,G_t
0,1,0,S0,Right,2.0,5.3300
1,1,1,S1,Up,1.0,3.7000
2,1,2,S2,Right,3.0,3.0000
3,2,0,S0,Left,-2.0,-0.4520
4,2,1,S1,Right,1.0,1.7200
5,2,2,S2,Down,-1.0,0.8000
6,2,3,S3,Right,2.0,2.0000
7,3,0,S0,Right,1.0,4.1734
8,3,1,S1,Left,-2.0,3.5260
9,3,2,S2,Up,2.0,6.1400



# 4. REINFORCE Policy Gradient

**How it works:** REINFORCE finishes one episode, computes the Monte Carlo return $G_t$ for every step, and then updates the policy using those returns.

$$
\theta
\leftarrow
\theta
+
\alpha\gamma^tG_t
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$

For the softmax logits,

$$
\mathbf g_t
=
\mathbf e_{A_t}
-
\boldsymbol{\pi}(S_t).
$$

### Algorithm

```text
For each episode:
    Generate one complete trajectory.
    Compute G_t for all steps.
    For each stored step t:
        Compute g_t.
        Update the policy with α γ^t G_t g_t.
```

### Full calculations


## Trajectory 1

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\gamma^{0}G_0=(1.0000)(5.3300)=5.3300
$$


$$
\boldsymbol{\pi}(S0)=\begin{bmatrix}0.2500\\0.2500\\0.2500\\0.2500\end{bmatrix}
$$


$$
\mathbf g_0=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S0)=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{0}G_0\mathbf g_0\\&=0.1(5.3300)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1333\\-0.1333\\-0.1333\\0.3998\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.0000\\0.0000\\0.0000\\0.0000\end{bmatrix}+\begin{bmatrix}-0.1333\\-0.1333\\-0.1333\\0.3998\end{bmatrix}=\begin{bmatrix}-0.1333\\-0.1333\\-0.1333\\0.3998\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2126\\0.2126\\0.2126\\0.3622\end{bmatrix}
$$


### Step t = 1: state S1, action Up

$$
\gamma^{1}G_1=(0.9000)(3.7000)=3.3300
$$


$$
\boldsymbol{\pi}(S1)=\begin{bmatrix}0.2500\\0.2500\\0.2500\\0.2500\end{bmatrix}
$$


$$
\mathbf g_1=\mathbf e_{\mathrm{Up}}-\boldsymbol{\pi}(S1)=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{1}G_1\mathbf g_1\\&=0.1(3.3300)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.2498\\-0.0833\\-0.0833\\-0.0833\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.0000\\0.0000\\0.0000\\0.0000\end{bmatrix}+\begin{bmatrix}0.2498\\-0.0833\\-0.0833\\-0.0833\end{bmatrix}=\begin{bmatrix}0.2498\\-0.0833\\-0.0833\\-0.0833\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.3174\\0.2275\\0.2275\\0.2275\end{bmatrix}
$$


### Step t = 2: state S2, action Right

$$
\gamma^{2}G_2=(0.8100)(3.0000)=2.4300
$$


$$
\boldsymbol{\pi}(S2)=\begin{bmatrix}0.2500\\0.2500\\0.2500\\0.2500\end{bmatrix}
$$


$$
\mathbf g_2=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S2)=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{2}G_2\mathbf g_2\\&=0.1(2.4300)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0608\\-0.0608\\-0.0608\\0.1823\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.0000\\0.0000\\0.0000\\0.0000\end{bmatrix}+\begin{bmatrix}-0.0608\\-0.0608\\-0.0608\\0.1823\end{bmatrix}=\begin{bmatrix}-0.0608\\-0.0608\\-0.0608\\0.1823\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2339\\0.2339\\0.2339\\0.2983\end{bmatrix}
$$


## Trajectory 2

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Left

$$
\gamma^{0}G_0=(1.0000)(-0.4520)=-0.4520
$$


$$
\boldsymbol{\pi}(S0)=\begin{bmatrix}0.2126\\0.2126\\0.2126\\0.3622\end{bmatrix}
$$


$$
\mathbf g_0=\mathbf e_{\mathrm{Left}}-\boldsymbol{\pi}(S0)=\begin{bmatrix}-0.2126\\-0.2126\\0.7874\\-0.3622\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{0}G_0\mathbf g_0\\&=0.1(-0.4520)\begin{bmatrix}-0.2126\\-0.2126\\0.7874\\-0.3622\end{bmatrix}\\&=\begin{bmatrix}0.0096\\0.0096\\-0.0356\\0.0164\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}-0.1333\\-0.1333\\-0.1333\\0.3998\end{bmatrix}+\begin{bmatrix}0.0096\\0.0096\\-0.0356\\0.0164\end{bmatrix}=\begin{bmatrix}-0.1236\\-0.1236\\-0.1688\\0.4161\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2141\\0.2141\\0.2046\\0.3673\end{bmatrix}
$$


### Step t = 1: state S1, action Right

$$
\gamma^{1}G_1=(0.9000)(1.7200)=1.5480
$$


$$
\boldsymbol{\pi}(S1)=\begin{bmatrix}0.3174\\0.2275\\0.2275\\0.2275\end{bmatrix}
$$


$$
\mathbf g_1=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S1)=\begin{bmatrix}-0.3174\\-0.2275\\-0.2275\\0.7725\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{1}G_1\mathbf g_1\\&=0.1(1.5480)\begin{bmatrix}-0.3174\\-0.2275\\-0.2275\\0.7725\end{bmatrix}\\&=\begin{bmatrix}-0.0491\\-0.0352\\-0.0352\\0.1196\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.2498\\-0.0833\\-0.0833\\-0.0833\end{bmatrix}+\begin{bmatrix}-0.0491\\-0.0352\\-0.0352\\0.1196\end{bmatrix}=\begin{bmatrix}0.2006\\-0.1185\\-0.1185\\0.0363\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.3028\\0.2201\\0.2201\\0.2570\end{bmatrix}
$$


### Step t = 2: state S2, action Down

$$
\gamma^{2}G_2=(0.8100)(0.8000)=0.6480
$$


$$
\boldsymbol{\pi}(S2)=\begin{bmatrix}0.2339\\0.2339\\0.2339\\0.2983\end{bmatrix}
$$


$$
\mathbf g_2=\mathbf e_{\mathrm{Down}}-\boldsymbol{\pi}(S2)=\begin{bmatrix}-0.2339\\0.7661\\-0.2339\\-0.2983\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{2}G_2\mathbf g_2\\&=0.1(0.6480)\begin{bmatrix}-0.2339\\0.7661\\-0.2339\\-0.2983\end{bmatrix}\\&=\begin{bmatrix}-0.0152\\0.0496\\-0.0152\\-0.0193\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}-0.0608\\-0.0608\\-0.0608\\0.1823\end{bmatrix}+\begin{bmatrix}-0.0152\\0.0496\\-0.0152\\-0.0193\end{bmatrix}=\begin{bmatrix}-0.0759\\-0.0111\\-0.0759\\0.1629\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2306\\0.2460\\0.2306\\0.2928\end{bmatrix}
$$


### Step t = 3: state S3, action Right

$$
\gamma^{3}G_3=(0.7290)(2.0000)=1.4580
$$


$$
\boldsymbol{\pi}(S3)=\begin{bmatrix}0.2500\\0.2500\\0.2500\\0.2500\end{bmatrix}
$$


$$
\mathbf g_3=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S3)=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{3}G_3\mathbf g_3\\&=0.1(1.4580)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0365\\-0.0365\\-0.0365\\0.1094\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.0000\\0.0000\\0.0000\\0.0000\end{bmatrix}+\begin{bmatrix}-0.0365\\-0.0365\\-0.0365\\0.1094\end{bmatrix}=\begin{bmatrix}-0.0365\\-0.0365\\-0.0365\\0.1094\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2406\\0.2406\\0.2406\\0.2783\end{bmatrix}
$$


## Trajectory 3

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\gamma^{0}G_0=(1.0000)(4.1734)=4.1734
$$


$$
\boldsymbol{\pi}(S0)=\begin{bmatrix}0.2141\\0.2141\\0.2046\\0.3673\end{bmatrix}
$$


$$
\mathbf g_0=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S0)=\begin{bmatrix}-0.2141\\-0.2141\\-0.2046\\0.6327\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{0}G_0\mathbf g_0\\&=0.1(4.1734)\begin{bmatrix}-0.2141\\-0.2141\\-0.2046\\0.6327\end{bmatrix}\\&=\begin{bmatrix}-0.0893\\-0.0893\\-0.0854\\0.2641\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}-0.1236\\-0.1236\\-0.1688\\0.4161\end{bmatrix}+\begin{bmatrix}-0.0893\\-0.0893\\-0.0854\\0.2641\end{bmatrix}=\begin{bmatrix}-0.2130\\-0.2130\\-0.2542\\0.6802\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.1851\\0.1851\\0.1776\\0.4522\end{bmatrix}
$$


### Step t = 1: state S1, action Left

$$
\gamma^{1}G_1=(0.9000)(3.5260)=3.1734
$$


$$
\boldsymbol{\pi}(S1)=\begin{bmatrix}0.3028\\0.2201\\0.2201\\0.2570\end{bmatrix}
$$


$$
\mathbf g_1=\mathbf e_{\mathrm{Left}}-\boldsymbol{\pi}(S1)=\begin{bmatrix}-0.3028\\-0.2201\\0.7799\\-0.2570\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{1}G_1\mathbf g_1\\&=0.1(3.1734)\begin{bmatrix}-0.3028\\-0.2201\\0.7799\\-0.2570\end{bmatrix}\\&=\begin{bmatrix}-0.0961\\-0.0698\\0.2475\\-0.0815\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.2006\\-0.1185\\-0.1185\\0.0363\end{bmatrix}+\begin{bmatrix}-0.0961\\-0.0698\\0.2475\\-0.0815\end{bmatrix}=\begin{bmatrix}0.1045\\-0.1883\\0.1290\\-0.0452\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.2753\\0.2054\\0.2822\\0.2371\end{bmatrix}
$$


### Step t = 2: state S2, action Up

$$
\gamma^{2}G_2=(0.8100)(6.1400)=4.9734
$$


$$
\boldsymbol{\pi}(S2)=\begin{bmatrix}0.2306\\0.2460\\0.2306\\0.2928\end{bmatrix}
$$


$$
\mathbf g_2=\mathbf e_{\mathrm{Up}}-\boldsymbol{\pi}(S2)=\begin{bmatrix}0.7694\\-0.2460\\-0.2306\\-0.2928\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{2}G_2\mathbf g_2\\&=0.1(4.9734)\begin{bmatrix}0.7694\\-0.2460\\-0.2306\\-0.2928\end{bmatrix}\\&=\begin{bmatrix}0.3827\\-0.1224\\-0.1147\\-0.1456\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}-0.0759\\-0.0111\\-0.0759\\0.1629\end{bmatrix}+\begin{bmatrix}0.3827\\-0.1224\\-0.1147\\-0.1456\end{bmatrix}=\begin{bmatrix}0.3068\\-0.1335\\-0.1906\\0.0173\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.3333\\0.2146\\0.2027\\0.2495\end{bmatrix}
$$


### Step t = 3: state S3, action Down

$$
\gamma^{3}G_3=(0.7290)(4.6000)=3.3534
$$


$$
\boldsymbol{\pi}(S3)=\begin{bmatrix}0.2406\\0.2406\\0.2406\\0.2783\end{bmatrix}
$$


$$
\mathbf g_3=\mathbf e_{\mathrm{Down}}-\boldsymbol{\pi}(S3)=\begin{bmatrix}-0.2406\\0.7594\\-0.2406\\-0.2783\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{3}G_3\mathbf g_3\\&=0.1(3.3534)\begin{bmatrix}-0.2406\\0.7594\\-0.2406\\-0.2783\end{bmatrix}\\&=\begin{bmatrix}-0.0807\\0.2547\\-0.0807\\-0.0933\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}-0.0365\\-0.0365\\-0.0365\\0.1094\end{bmatrix}+\begin{bmatrix}-0.0807\\0.2547\\-0.0807\\-0.0933\end{bmatrix}=\begin{bmatrix}-0.1171\\0.2182\\-0.1171\\0.0160\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2202\\0.3080\\0.2202\\0.2516\end{bmatrix}
$$


### Step t = 4: state S4, action Right

$$
\gamma^{4}G_4=(0.6561)(4.0000)=2.6244
$$


$$
\boldsymbol{\pi}(S4)=\begin{bmatrix}0.2500\\0.2500\\0.2500\\0.2500\end{bmatrix}
$$


$$
\mathbf g_4=\mathbf e_{\mathrm{Right}}-\boldsymbol{\pi}(S4)=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\Delta\mathbf z&=\alpha\gamma^{4}G_4\mathbf g_4\\&=0.1(2.6244)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0656\\-0.0656\\-0.0656\\0.1968\end{bmatrix}\end{aligned}
$$


$$
\mathbf z_{new}=\begin{bmatrix}0.0000\\0.0000\\0.0000\\0.0000\end{bmatrix}+\begin{bmatrix}-0.0656\\-0.0656\\-0.0656\\0.1968\end{bmatrix}=\begin{bmatrix}-0.0656\\-0.0656\\-0.0656\\0.1968\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S4)=\begin{bmatrix}0.2326\\0.2326\\0.2326\\0.3023\end{bmatrix}
$$


### How to read the REINFORCE update

REINFORCE uses

$$
\theta
\leftarrow
\theta
+
\alpha\gamma^tG_t
\nabla_{\theta}
\log\pi_{\theta}(A_t\mid S_t).
$$

Meaning:

- $\theta$: current policy parameters;
- $\alpha$: learning rate or step size;
- $t$: current position inside the completed trajectory;
- $\gamma^t$: discount applied to the contribution of time step $t$ under the discounted episodic objective used in these slides;
- $G_t$: complete Monte Carlo return after step $t$;
- $\pi_{\theta}(A_t\mid S_t)$: probability assigned by the current policy to the action that was actually selected;
- $\log\pi_{\theta}(A_t\mid S_t)$: log-probability of that selected action;
- $\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t)$: direction that most increases the log-probability of the selected action.

**How to read the formula:** multiply the selected action's policy gradient by the return that followed that action. A positive return pushes the policy toward the sampled action; a negative return pushes it away.

For one completed trajectory,

$$
\hat{\mathbf g}_{\mathrm{REINFORCE}}
=
\sum_{t=0}^{T-1}
\gamma^tG_t
\nabla_{\theta}
\log\pi_{\theta}(A_t\mid S_t).
$$

Here the summation is **over time steps inside one trajectory**.

If many trajectories were averaged, then an additional outer average over trajectories could also be used. The slide-style REINFORCE calculation in this notebook processes one completed trajectory at a time.

In [8]:

# REINFORCE verification
states = ["S0","S1","S2","S3","S4"]
logits = {s:np.zeros(4) for s in states}

for traj in trajectories:
    Gs = discounted_returns(traj)
    for t,((s,a,r),G) in enumerate(zip(traj,Gs)):
        p = softmax(logits[s])
        logits[s] += alpha*(gamma**t)*G*grad_log_softmax(p,a)

pd.DataFrame({s:softmax(logits[s]) for s in states}, index=ACTIONS).T


,Up,Down,Left,Right
S0,0.1851,0.1851,0.1776,0.4522
S1,0.2753,0.2054,0.2822,0.2371
S2,0.3333,0.2146,0.2027,0.2495
S3,0.2202,0.3080,0.2202,0.2516
S4,0.2326,0.2326,0.2326,0.3023



# 5. Vanilla Policy Gradient

**How it works:** Here, Vanilla PG collects all three trajectories with the same fixed policy, computes the trajectory-gradient contributions, averages them, and then updates once.

With $N=3$,

$$
\hat{\mathbf g}_{\mathrm{VPG}}
=
\frac{1}{3}
\sum_{i=1}^{3}
\sum_{t=0}^{T_i-1}
\gamma^tG_t^{(i)}
\nabla_{\theta}\log\pi_{\theta}(A_t^{(i)}\mid S_t^{(i)}).
$$

Then

$$
\theta
\leftarrow
\theta+\alpha\hat{\mathbf g}_{\mathrm{VPG}}.
$$

The policy stays fixed while the three trajectories are collected:

$$
\boldsymbol{\pi}(S_t)=
\begin{bmatrix}
0.25\\0.25\\0.25\\0.25
\end{bmatrix}.
$$

### Algorithm

```text
Keep θ fixed.
Collect trajectory 1, 2, and 3.
Compute all trajectory-gradient contributions.
Average the three trajectory gradients.
Update θ once.
```

### Full calculations


## Trajectory 1

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\gamma^{0}G_t=(1.0000)(5.3300)=5.3300
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}G_t\mathbf g_t\\&=\frac{1}{3}(5.3300)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.4442\\-0.4442\\-0.4442\\1.3325\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Up

$$
\gamma^{1}G_t=(0.9000)(3.7000)=3.3300
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Up}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}G_t\mathbf g_t\\&=\frac{1}{3}(3.3300)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.8325\\-0.2775\\-0.2775\\-0.2775\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Right

$$
\gamma^{2}G_t=(0.8100)(3.0000)=2.4300
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}G_t\mathbf g_t\\&=\frac{1}{3}(2.4300)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2025\\-0.2025\\-0.2025\\0.6075\end{bmatrix}\end{aligned}
$$


## Trajectory 2

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Left

$$
\gamma^{0}G_t=(1.0000)(-0.4520)=-0.4520
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Left}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}G_t\mathbf g_t\\&=\frac{1}{3}(-0.4520)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.0377\\0.0377\\-0.1130\\0.0377\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Right

$$
\gamma^{1}G_t=(0.9000)(1.7200)=1.5480
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}G_t\mathbf g_t\\&=\frac{1}{3}(1.5480)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1290\\-0.1290\\-0.1290\\0.3870\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Down

$$
\gamma^{2}G_t=(0.8100)(0.8000)=0.6480
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Down}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}G_t\mathbf g_t\\&=\frac{1}{3}(0.6480)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.0540\\0.1620\\-0.0540\\-0.0540\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Right

$$
\gamma^{3}G_t=(0.7290)(2.0000)=1.4580
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}G_t\mathbf g_t\\&=\frac{1}{3}(1.4580)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1215\\-0.1215\\-0.1215\\0.3645\end{bmatrix}\end{aligned}
$$


## Trajectory 3

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\gamma^{0}G_t=(1.0000)(4.1734)=4.1734
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}G_t\mathbf g_t\\&=\frac{1}{3}(4.1734)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.3478\\-0.3478\\-0.3478\\1.0433\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Left

$$
\gamma^{1}G_t=(0.9000)(3.5260)=3.1734
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Left}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}G_t\mathbf g_t\\&=\frac{1}{3}(3.1734)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.2645\\-0.2645\\0.7933\\-0.2645\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Up

$$
\gamma^{2}G_t=(0.8100)(6.1400)=4.9734
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Up}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}G_t\mathbf g_t\\&=\frac{1}{3}(4.9734)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}1.2433\\-0.4144\\-0.4144\\-0.4144\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Down

$$
\gamma^{3}G_t=(0.7290)(4.6000)=3.3534
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Down}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}G_t\mathbf g_t\\&=\frac{1}{3}(3.3534)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.2795\\0.8384\\-0.2795\\-0.2795\end{bmatrix}\end{aligned}
$$


### Step t = 4: state S4, action Right

$$
\gamma^{4}G_t=(0.6561)(4.0000)=2.6244
$$


$$
\mathbf g_t=\mathbf e_{\mathrm{Right}}-\begin{bmatrix}0.25\\0.25\\0.25\\0.25\end{bmatrix}=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{4}G_t\mathbf g_t\\&=\frac{1}{3}(2.6244)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}\end{aligned}
$$


## Average the three trajectories and update once

### State S0

$$
\hat{\mathbf g}(S0)=\begin{bmatrix}-0.7543\\-0.7543\\-0.9049\\2.4135\end{bmatrix}
$$


$$
\mathbf z_{new}(S0)=0.1\begin{bmatrix}-0.7543\\-0.7543\\-0.9049\\2.4135\end{bmatrix}=\begin{bmatrix}-0.0754\\-0.0754\\-0.0905\\0.2414\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2295\\0.2295\\0.2260\\0.3150\end{bmatrix}
$$


### State S1

$$
\hat{\mathbf g}(S1)=\begin{bmatrix}0.4390\\-0.6710\\0.3868\\-0.1549\end{bmatrix}
$$


$$
\mathbf z_{new}(S1)=0.1\begin{bmatrix}0.4390\\-0.6710\\0.3868\\-0.1549\end{bmatrix}=\begin{bmatrix}0.0439\\-0.0671\\0.0387\\-0.0155\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.2610\\0.2335\\0.2596\\0.2459\end{bmatrix}
$$


### State S2

$$
\hat{\mathbf g}(S2)=\begin{bmatrix}0.9869\\-0.4549\\-0.6709\\0.1391\end{bmatrix}
$$


$$
\mathbf z_{new}(S2)=0.1\begin{bmatrix}0.9869\\-0.4549\\-0.6709\\0.1391\end{bmatrix}=\begin{bmatrix}0.0987\\-0.0455\\-0.0671\\0.0139\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2754\\0.2384\\0.2333\\0.2530\end{bmatrix}
$$


### State S3

$$
\hat{\mathbf g}(S3)=\begin{bmatrix}-0.4010\\0.7168\\-0.4010\\0.0851\end{bmatrix}
$$


$$
\mathbf z_{new}(S3)=0.1\begin{bmatrix}-0.4010\\0.7168\\-0.4010\\0.0851\end{bmatrix}=\begin{bmatrix}-0.0401\\0.0717\\-0.0401\\0.0085\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2399\\0.2683\\0.2399\\0.2519\end{bmatrix}
$$


### State S4

$$
\hat{\mathbf g}(S4)=\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}
$$


$$
\mathbf z_{new}(S4)=0.1\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}=\begin{bmatrix}-0.0219\\-0.0219\\-0.0219\\0.0656\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S4)=\begin{bmatrix}0.2444\\0.2444\\0.2444\\0.2668\end{bmatrix}
$$


### How to read the Vanilla Policy Gradient equation

The theoretical form is

$$
\nabla_{\theta}J(\theta)
=
\mathbb E_{\tau\sim\pi_{\theta}}
\left[
\sum_{t=0}^{T-1}
\gamma^tG_t
\nabla_{\theta}
\log\pi_{\theta}(A_t\mid S_t)
\right].
$$

**How to read it:** for each possible trajectory generated by the current policy, sum all time-step contributions; then take the expected value across those trajectories.

The expectation is approximated in this notebook with three sampled trajectories:

$$
\hat{\mathbf g}_{\mathrm{VPG}}
=
\frac{1}{3}
\sum_{i=1}^{3}
\sum_{t=0}^{T_i-1}
\gamma^tG_t^{(i)}
\nabla_{\theta}
\log\pi_{\theta}
\left(
A_t^{(i)}\mid S_t^{(i)}
\right).
$$

Meaning:

- $i$: trajectory index;
- $N=3$: number of trajectories in the batch;
- $T_i$: number of decisions in trajectory $i$;
- $G_t^{(i)}$: Monte Carlo return at step $t$ of trajectory $i$;
- inner $\sum_t$: sum over steps inside one trajectory;
- outer $\sum_i$: sum over the three sampled trajectories;
- $1/3$: average across the three trajectories.

The policy parameters are kept fixed while these three trajectories are collected. After their gradients are averaged, one policy update is made.

In [9]:

# Vanilla PG verification
p0 = np.ones(4)/4
N = len(trajectories)
grad_sum = {s:np.zeros(4) for s in states}

for traj in trajectories:
    Gs = discounted_returns(traj)
    for t,((s,a,r),G) in enumerate(zip(traj,Gs)):
        grad_sum[s] += ((gamma**t)*G*grad_log_softmax(p0,a))/N

vanilla_logits = {s:alpha*grad_sum[s] for s in states}
pd.DataFrame({s:softmax(vanilla_logits[s]) for s in states}, index=ACTIONS).T


,Up,Down,Left,Right
S0,0.2295,0.2295,0.2260,0.3150
S1,0.2610,0.2335,0.2596,0.2459
S2,0.2754,0.2384,0.2333,0.2530
S3,0.2399,0.2683,0.2399,0.2519
S4,0.2444,0.2444,0.2444,0.2668



# 6. Advantage Policy Gradient

**How it works:** Advantage PG compares the selected action with the baseline value of the current state.

$$
A(S_t,A_t)=Q(S_t,A_t)-V(S_t).
$$

The gradient estimate is

$$
\hat{\mathbf g}_{A}
=
\frac{1}{3}
\sum_i
\sum_t
\gamma^tA_t
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$

Positive advantage encourages the sampled action; negative advantage discourages it.

### Critic values used here

| State | Up | Down | Left | Right | V(s) |
|---|---:|---:|---:|---:|---:|
| S0 | 0.50 | -0.50 | -2.00 | 3.00 | 0.25 |
| S1 | 3.00 | 0.00 | -2.00 | 1.00 | 0.50 |
| S2 | 2.00 | -1.00 | 0.00 | 3.00 | 1.00 |
| S3 | 0.00 | 2.50 | -1.00 | 1.50 | 0.75 |
| S4 | 1.00 | -1.00 | 0.00 | 4.00 | 1.00 |

### Full calculations

## Trajectory 1

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\begin{aligned}A_t&=Q(S0,\mathrm{Right})-V(S0)\\&=3.0000-0.2500\\&=2.7500\end{aligned}
$$


$$
\gamma^{0}A_t=(1.0000)(2.7500)=2.7500
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}A_t\mathbf g_t\\&=\frac{1}{3}(2.7500)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2292\\-0.2292\\-0.2292\\0.6875\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Up

$$
\begin{aligned}A_t&=Q(S1,\mathrm{Up})-V(S1)\\&=3.0000-0.5000\\&=2.5000\end{aligned}
$$


$$
\gamma^{1}A_t=(0.9000)(2.5000)=2.2500
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}A_t\mathbf g_t\\&=\frac{1}{3}(2.2500)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.5625\\-0.1875\\-0.1875\\-0.1875\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Right

$$
\begin{aligned}A_t&=Q(S2,\mathrm{Right})-V(S2)\\&=3.0000-1.0000\\&=2.0000\end{aligned}
$$


$$
\gamma^{2}A_t=(0.8100)(2.0000)=1.6200
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}A_t\mathbf g_t\\&=\frac{1}{3}(1.6200)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1350\\-0.1350\\-0.1350\\0.4050\end{bmatrix}\end{aligned}
$$


## Trajectory 2

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Left

$$
\begin{aligned}A_t&=Q(S0,\mathrm{Left})-V(S0)\\&=-2.0000-0.2500\\&=-2.2500\end{aligned}
$$


$$
\gamma^{0}A_t=(1.0000)(-2.2500)=-2.2500
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}A_t\mathbf g_t\\&=\frac{1}{3}(-2.2500)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1875\\0.1875\\-0.5625\\0.1875\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Right

$$
\begin{aligned}A_t&=Q(S1,\mathrm{Right})-V(S1)\\&=1.0000-0.5000\\&=0.5000\end{aligned}
$$


$$
\gamma^{1}A_t=(0.9000)(0.5000)=0.4500
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}A_t\mathbf g_t\\&=\frac{1}{3}(0.4500)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0375\\-0.0375\\-0.0375\\0.1125\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Down

$$
\begin{aligned}A_t&=Q(S2,\mathrm{Down})-V(S2)\\&=-1.0000-1.0000\\&=-2.0000\end{aligned}
$$


$$
\gamma^{2}A_t=(0.8100)(-2.0000)=-1.6200
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}A_t\mathbf g_t\\&=\frac{1}{3}(-1.6200)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1350\\-0.4050\\0.1350\\0.1350\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Right

$$
\begin{aligned}A_t&=Q(S3,\mathrm{Right})-V(S3)\\&=1.5000-0.7500\\&=0.7500\end{aligned}
$$


$$
\gamma^{3}A_t=(0.7290)(0.7500)=0.5468
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}A_t\mathbf g_t\\&=\frac{1}{3}(0.5468)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0456\\-0.0456\\-0.0456\\0.1367\end{bmatrix}\end{aligned}
$$


## Trajectory 3

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\begin{aligned}A_t&=Q(S0,\mathrm{Right})-V(S0)\\&=3.0000-0.2500\\&=2.7500\end{aligned}
$$


$$
\gamma^{0}A_t=(1.0000)(2.7500)=2.7500
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}A_t\mathbf g_t\\&=\frac{1}{3}(2.7500)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2292\\-0.2292\\-0.2292\\0.6875\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Left

$$
\begin{aligned}A_t&=Q(S1,\mathrm{Left})-V(S1)\\&=-2.0000-0.5000\\&=-2.5000\end{aligned}
$$


$$
\gamma^{1}A_t=(0.9000)(-2.5000)=-2.2500
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}A_t\mathbf g_t\\&=\frac{1}{3}(-2.2500)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1875\\0.1875\\-0.5625\\0.1875\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Up

$$
\begin{aligned}A_t&=Q(S2,\mathrm{Up})-V(S2)\\&=2.0000-1.0000\\&=1.0000\end{aligned}
$$


$$
\gamma^{2}A_t=(0.8100)(1.0000)=0.8100
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}A_t\mathbf g_t\\&=\frac{1}{3}(0.8100)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.2025\\-0.0675\\-0.0675\\-0.0675\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Down

$$
\begin{aligned}A_t&=Q(S3,\mathrm{Down})-V(S3)\\&=2.5000-0.7500\\&=1.7500\end{aligned}
$$


$$
\gamma^{3}A_t=(0.7290)(1.7500)=1.2758
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}A_t\mathbf g_t\\&=\frac{1}{3}(1.2758)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.1063\\0.3189\\-0.1063\\-0.1063\end{bmatrix}\end{aligned}
$$


### Step t = 4: state S4, action Right

$$
\begin{aligned}A_t&=Q(S4,\mathrm{Right})-V(S4)\\&=4.0000-1.0000\\&=3.0000\end{aligned}
$$


$$
\gamma^{4}A_t=(0.6561)(3.0000)=1.9683
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{4}A_t\mathbf g_t\\&=\frac{1}{3}(1.9683)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}\end{aligned}
$$


## Average the three trajectories and update once

### State S0

$$
\hat{\mathbf g}_A(S0)=\begin{bmatrix}-0.2708\\-0.2708\\-1.0208\\1.5625\end{bmatrix}
$$


$$
\mathbf z_{new}(S0)=0.1\begin{bmatrix}-0.2708\\-0.2708\\-1.0208\\1.5625\end{bmatrix}=\begin{bmatrix}-0.0271\\-0.0271\\-0.1021\\0.1562\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2422\\0.2422\\0.2247\\0.2909\end{bmatrix}
$$


### State S1

$$
\hat{\mathbf g}_A(S1)=\begin{bmatrix}0.7125\\-0.0375\\-0.7875\\0.1125\end{bmatrix}
$$


$$
\mathbf z_{new}(S1)=0.1\begin{bmatrix}0.7125\\-0.0375\\-0.7875\\0.1125\end{bmatrix}=\begin{bmatrix}0.0713\\-0.0038\\-0.0788\\0.0113\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.2681\\0.2487\\0.2307\\0.2525\end{bmatrix}
$$


### State S2

$$
\hat{\mathbf g}_A(S2)=\begin{bmatrix}0.2025\\-0.6075\\-0.0675\\0.4725\end{bmatrix}
$$


$$
\mathbf z_{new}(S2)=0.1\begin{bmatrix}0.2025\\-0.6075\\-0.0675\\0.4725\end{bmatrix}=\begin{bmatrix}0.0203\\-0.0608\\-0.0068\\0.0473\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2549\\0.2351\\0.2481\\0.2619\end{bmatrix}
$$


### State S3

$$
\hat{\mathbf g}_A(S3)=\begin{bmatrix}-0.1519\\0.2734\\-0.1519\\0.0304\end{bmatrix}
$$


$$
\mathbf z_{new}(S3)=0.1\begin{bmatrix}-0.1519\\0.2734\\-0.1519\\0.0304\end{bmatrix}=\begin{bmatrix}-0.0152\\0.0273\\-0.0152\\0.0030\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2462\\0.2569\\0.2462\\0.2507\end{bmatrix}
$$


### State S4

$$
\hat{\mathbf g}_A(S4)=\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}
$$


$$
\mathbf z_{new}(S4)=0.1\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}=\begin{bmatrix}-0.0164\\-0.0164\\-0.0164\\0.0492\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S4)=\begin{bmatrix}0.2458\\0.2458\\0.2458\\0.2625\end{bmatrix}
$$


### How to read the Advantage Policy Gradient equations

The advantage is

$$
A(S_t,A_t)
=
Q(S_t,A_t)-V(S_t).
$$

Meaning:

- $Q(S_t,A_t)$: expected future return if action $A_t$ is taken in state $S_t$, then the policy continues;
- $V(S_t)$: expected future return from state $S_t$ under the policy;
- $A(S_t,A_t)$: how much better or worse the selected action is than the state's baseline value.

**How to read it:**

- $A_t>0$: the action is better than expected for this state;
- $A_t<0$: the action is worse than expected;
- $A_t\approx0$: the action is about as good as the baseline.

The batch policy-gradient estimate is

$$
\hat{\mathbf g}_{A}
=
\frac{1}{3}
\sum_{i=1}^{3}
\sum_{t=0}^{T_i-1}
\gamma^tA_t^{(i)}
\nabla_{\theta}
\log\pi_{\theta}
\left(
A_t^{(i)}\mid S_t^{(i)}
\right).
$$

**How to read it:** for each step, replace the raw Monte Carlo-return weight with an advantage weight; sum within each trajectory; then average across the three trajectories.

Here, $A_t^{(i)}$ appearing outside the log is the **advantage value**. The $A_t^{(i)}$ inside $\pi_{\theta}(A_t^{(i)}\mid S_t^{(i)})$ denotes the **selected action**. The same letter is traditional notation, but the roles are different; context tells which meaning is intended.

In [10]:

# Advantage PG verification
Q = {
    "S0":{"Up":0.5,"Down":-0.5,"Left":-2.0,"Right":3.0},
    "S1":{"Up":3.0,"Down":0.0,"Left":-2.0,"Right":1.0},
    "S2":{"Up":2.0,"Down":-1.0,"Left":0.0,"Right":3.0},
    "S3":{"Up":0.0,"Down":2.5,"Left":-1.0,"Right":1.5},
    "S4":{"Up":1.0,"Down":-1.0,"Left":0.0,"Right":4.0}
}
V = {s:np.mean([Q[s][a] for a in ACTIONS]) for s in states}

grad_adv = {s:np.zeros(4) for s in states}
for traj in trajectories:
    for t,(s,a,r) in enumerate(traj):
        A = Q[s][a] - V[s]
        grad_adv[s] += ((gamma**t)*A*grad_log_softmax(p0,a))/N

adv_logits = {s:alpha*grad_adv[s] for s in states}
pd.DataFrame({s:softmax(adv_logits[s]) for s in states}, index=ACTIONS).T


,Up,Down,Left,Right
S0,0.2422,0.2422,0.2247,0.2909
S1,0.2681,0.2487,0.2307,0.2525
S2,0.2549,0.2351,0.2481,0.2619
S3,0.2462,0.2569,0.2462,0.2507
S4,0.2458,0.2458,0.2458,0.2625



# 7. TD-Residual Policy Gradient

**How it works:** TD-residual PG uses a one-step bootstrapped error instead of a complete Monte Carlo return.

$$
\delta_t
=
R_{t+1}
+
\gamma V(S_{t+1})
-
V(S_t).
$$

For a terminal transition,

$$
V(\mathrm{Terminal})=0.
$$

The gradient estimate is

$$
\hat{\mathbf g}_{TD}
=
\frac{1}{3}
\sum_i
\sum_t
\gamma^t\delta_t
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$

### Full calculations


## Trajectory 1

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=2+0.9(0.5000)-0.2500\\&=2.2000\end{aligned}
$$


$$
\gamma^{0}\delta_t=(1.0000)(2.2000)=2.2000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}\delta_t\mathbf g_t\\&=\frac{1}{3}(2.2000)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1833\\-0.1833\\-0.1833\\0.5500\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Up

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=1+0.9(1.0000)-0.5000\\&=1.4000\end{aligned}
$$


$$
\gamma^{1}\delta_t=(0.9000)(1.4000)=1.2600
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.2600)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.3150\\-0.1050\\-0.1050\\-0.1050\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=3+0.9(0.0000)-1.0000\\&=2.0000\end{aligned}
$$


$$
\gamma^{2}\delta_t=(0.8100)(2.0000)=1.6200
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.6200)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1350\\-0.1350\\-0.1350\\0.4050\end{bmatrix}\end{aligned}
$$


## Trajectory 2

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Left

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=-2+0.9(0.5000)-0.2500\\&=-1.8000\end{aligned}
$$


$$
\gamma^{0}\delta_t=(1.0000)(-1.8000)=-1.8000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}\delta_t\mathbf g_t\\&=\frac{1}{3}(-1.8000)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1500\\0.1500\\-0.4500\\0.1500\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=1+0.9(1.0000)-0.5000\\&=1.4000\end{aligned}
$$


$$
\gamma^{1}\delta_t=(0.9000)(1.4000)=1.2600
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.2600)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1050\\-0.1050\\-0.1050\\0.3150\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Down

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=-1+0.9(0.7500)-1.0000\\&=-1.3250\end{aligned}
$$


$$
\gamma^{2}\delta_t=(0.8100)(-1.3250)=-1.0733
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}\delta_t\mathbf g_t\\&=\frac{1}{3}(-1.0733)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.0894\\-0.2683\\0.0894\\0.0894\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=2+0.9(0.0000)-0.7500\\&=1.2500\end{aligned}
$$


$$
\gamma^{3}\delta_t=(0.7290)(1.2500)=0.9113
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}\delta_t\mathbf g_t\\&=\frac{1}{3}(0.9113)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0759\\-0.0759\\-0.0759\\0.2278\end{bmatrix}\end{aligned}
$$


## Trajectory 3

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=1+0.9(0.5000)-0.2500\\&=1.2000\end{aligned}
$$


$$
\gamma^{0}\delta_t=(1.0000)(1.2000)=1.2000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.2000)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1000\\-0.1000\\-0.1000\\0.3000\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Left

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=-2+0.9(1.0000)-0.5000\\&=-1.6000\end{aligned}
$$


$$
\gamma^{1}\delta_t=(0.9000)(-1.6000)=-1.4400
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}\delta_t\mathbf g_t\\&=\frac{1}{3}(-1.4400)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1200\\0.1200\\-0.3600\\0.1200\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Up

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=2+0.9(0.7500)-1.0000\\&=1.6750\end{aligned}
$$


$$
\gamma^{2}\delta_t=(0.8100)(1.6750)=1.3567
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.3567)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.3392\\-0.1131\\-0.1131\\-0.1131\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Down

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=1+0.9(1.0000)-0.7500\\&=1.1500\end{aligned}
$$


$$
\gamma^{3}\delta_t=(0.7290)(1.1500)=0.8384
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}\delta_t\mathbf g_t\\&=\frac{1}{3}(0.8384)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.0699\\0.2096\\-0.0699\\-0.0699\end{bmatrix}\end{aligned}
$$


### Step t = 4: state S4, action Right

$$
\begin{aligned}\delta_t&=R_{t+1}+\gamma V(S_{t+1})-V(S_t)\\&=4+0.9(0.0000)-1.0000\\&=3.0000\end{aligned}
$$


$$
\gamma^{4}\delta_t=(0.6561)(3.0000)=1.9683
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{4}\delta_t\mathbf g_t\\&=\frac{1}{3}(1.9683)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}\end{aligned}
$$


## Average the three trajectories and update once

### State S0

$$
\hat{\mathbf g}_{TD}(S0)=\begin{bmatrix}-0.1333\\-0.1333\\-0.7333\\1.0000\end{bmatrix}
$$


$$
\mathbf z_{new}(S0)=0.1\begin{bmatrix}-0.1333\\-0.1333\\-0.7333\\1.0000\end{bmatrix}=\begin{bmatrix}-0.0133\\-0.0133\\-0.0733\\0.1000\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2462\\0.2462\\0.2319\\0.2757\end{bmatrix}
$$


### State S1

$$
\hat{\mathbf g}_{TD}(S1)=\begin{bmatrix}0.3300\\-0.0900\\-0.5700\\0.3300\end{bmatrix}
$$


$$
\mathbf z_{new}(S1)=0.1\begin{bmatrix}0.3300\\-0.0900\\-0.5700\\0.3300\end{bmatrix}=\begin{bmatrix}0.0330\\-0.0090\\-0.0570\\0.0330\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.2582\\0.2476\\0.2360\\0.2582\end{bmatrix}
$$


### State S2

$$
\hat{\mathbf g}_{TD}(S2)=\begin{bmatrix}0.2936\\-0.5164\\-0.1586\\0.3814\end{bmatrix}
$$


$$
\mathbf z_{new}(S2)=0.1\begin{bmatrix}0.2936\\-0.5164\\-0.1586\\0.3814\end{bmatrix}=\begin{bmatrix}0.0294\\-0.0516\\-0.0159\\0.0381\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2573\\0.2373\\0.2459\\0.2595\end{bmatrix}
$$


### State S3

$$
\hat{\mathbf g}_{TD}(S3)=\begin{bmatrix}-0.1458\\0.1336\\-0.1458\\0.1580\end{bmatrix}
$$


$$
\mathbf z_{new}(S3)=0.1\begin{bmatrix}-0.1458\\0.1336\\-0.1458\\0.1580\end{bmatrix}=\begin{bmatrix}-0.0146\\0.0134\\-0.0146\\0.0158\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2464\\0.2533\\0.2464\\0.2540\end{bmatrix}
$$


### State S4

$$
\hat{\mathbf g}_{TD}(S4)=\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}
$$


$$
\mathbf z_{new}(S4)=0.1\begin{bmatrix}-0.1640\\-0.1640\\-0.1640\\0.4921\end{bmatrix}=\begin{bmatrix}-0.0164\\-0.0164\\-0.0164\\0.0492\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S4)=\begin{bmatrix}0.2458\\0.2458\\0.2458\\0.2625\end{bmatrix}
$$


### How to read the TD-residual equations

The temporal-difference residual is

$$
\delta_t
=
R_{t+1}
+
\gamma V(S_{t+1})
-
V(S_t).
$$

Meaning:

- $R_{t+1}$: reward observed after taking $A_t$;
- $V(S_t)$: current estimate of the value of the present state;
- $V(S_{t+1})$: current estimate of the value of the next state;
- $\gamma V(S_{t+1})$: discounted estimate of future value;
- $\delta_t$: one-step prediction error.

**How to read it:** compare what the current state was expected to be worth, $V(S_t)$, with what was actually observed after one transition: immediate reward plus discounted next-state value.

For a terminal transition,

$$
V(\mathrm{Terminal})=0.
$$

The policy-gradient estimate is

$$
\hat{\mathbf g}_{TD}
=
\frac{1}{3}
\sum_{i=1}^{3}
\sum_{t=0}^{T_i-1}
\gamma^t\delta_t^{(i)}
\nabla_{\theta}
\log\pi_{\theta}
\left(
A_t^{(i)}\mid S_t^{(i)}
\right).
$$

**How to read it:** sum the TD-error-weighted policy gradients over the steps of each trajectory, then average the trajectory contributions.

Unlike Monte Carlo $G_t$, $\delta_t$ can be computed after observing only

$$
S_t,\;A_t,\;R_{t+1},\;S_{t+1}.
$$

That is why TD learning can update before the whole episode has finished.

In [11]:

# TD-residual PG verification
grad_td = {s:np.zeros(4) for s in states}

for traj in trajectories:
    for t,(s,a,r) in enumerate(traj):
        v_next = V[traj[t+1][0]] if t+1 < len(traj) else 0.0
        delta = r + gamma*v_next - V[s]
        grad_td[s] += ((gamma**t)*delta*grad_log_softmax(p0,a))/N

td_logits = {s:alpha*grad_td[s] for s in states}
pd.DataFrame({s:softmax(td_logits[s]) for s in states}, index=ACTIONS).T


,Up,Down,Left,Right
S0,0.2462,0.2462,0.2319,0.2757
S1,0.2582,0.2476,0.2360,0.2582
S2,0.2573,0.2373,0.2459,0.2595
S3,0.2464,0.2533,0.2464,0.2540
S4,0.2458,0.2458,0.2458,0.2625



# 8. Q-Weighted Policy Gradient

**How it works:** Q-weighted PG uses the critic's estimate $Q(S_t,A_t)$ directly as the learning signal.

$$
\hat{\mathbf g}_{Q}
=
\frac{1}{3}
\sum_i
\sum_t
\gamma^tQ(S_t,A_t)
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$

### Full calculations


## Trajectory 1

$$
S0\xrightarrow{\mathrm{Right},2}S1\xrightarrow{\mathrm{Up},1}S2\xrightarrow{\mathrm{Right},3}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
Q_t=Q(S0,\mathrm{Right})=3.0000
$$


$$
\gamma^{0}Q_t=(1.0000)(3.0000)=3.0000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}Q_t\mathbf g_t\\&=\frac{1}{3}(3.0000)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Up

$$
Q_t=Q(S1,\mathrm{Up})=3.0000
$$


$$
\gamma^{1}Q_t=(0.9000)(3.0000)=2.7000
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}Q_t\mathbf g_t\\&=\frac{1}{3}(2.7000)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.6750\\-0.2250\\-0.2250\\-0.2250\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Right

$$
Q_t=Q(S2,\mathrm{Right})=3.0000
$$


$$
\gamma^{2}Q_t=(0.8100)(3.0000)=2.4300
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}Q_t\mathbf g_t\\&=\frac{1}{3}(2.4300)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2025\\-0.2025\\-0.2025\\0.6075\end{bmatrix}\end{aligned}
$$


## Trajectory 2

$$
S0\xrightarrow{\mathrm{Left},-2}S1\xrightarrow{\mathrm{Right},1}S2\xrightarrow{\mathrm{Down},-1}S3\xrightarrow{\mathrm{Right},2}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Left

$$
Q_t=Q(S0,\mathrm{Left})=-2.0000
$$


$$
\gamma^{0}Q_t=(1.0000)(-2.0000)=-2.0000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}Q_t\mathbf g_t\\&=\frac{1}{3}(-2.0000)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1667\\0.1667\\-0.5000\\0.1667\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Right

$$
Q_t=Q(S1,\mathrm{Right})=1.0000
$$


$$
\gamma^{1}Q_t=(0.9000)(1.0000)=0.9000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}Q_t\mathbf g_t\\&=\frac{1}{3}(0.9000)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0750\\-0.0750\\-0.0750\\0.2250\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Down

$$
Q_t=Q(S2,\mathrm{Down})=-1.0000
$$


$$
\gamma^{2}Q_t=(0.8100)(-1.0000)=-0.8100
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}Q_t\mathbf g_t\\&=\frac{1}{3}(-0.8100)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.0675\\-0.2025\\0.0675\\0.0675\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Right

$$
Q_t=Q(S3,\mathrm{Right})=1.5000
$$


$$
\gamma^{3}Q_t=(0.7290)(1.5000)=1.0935
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}Q_t\mathbf g_t\\&=\frac{1}{3}(1.0935)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.0911\\-0.0911\\-0.0911\\0.2734\end{bmatrix}\end{aligned}
$$


## Trajectory 3

$$
S0\xrightarrow{\mathrm{Right},1}S1\xrightarrow{\mathrm{Left},-2}S2\xrightarrow{\mathrm{Up},2}S3\xrightarrow{\mathrm{Down},1}S4\xrightarrow{\mathrm{Right},4}\mathrm{Terminal}
$$


### Step t = 0: state S0, action Right

$$
Q_t=Q(S0,\mathrm{Right})=3.0000
$$


$$
\gamma^{0}Q_t=(1.0000)(3.0000)=3.0000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{0}Q_t\mathbf g_t\\&=\frac{1}{3}(3.0000)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\end{aligned}
$$


### Step t = 1: state S1, action Left

$$
Q_t=Q(S1,\mathrm{Left})=-2.0000
$$


$$
\gamma^{1}Q_t=(0.9000)(-2.0000)=-1.8000
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{1}Q_t\mathbf g_t\\&=\frac{1}{3}(-1.8000)\begin{bmatrix}-0.2500\\-0.2500\\0.7500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.1500\\0.1500\\-0.4500\\0.1500\end{bmatrix}\end{aligned}
$$


### Step t = 2: state S2, action Up

$$
Q_t=Q(S2,\mathrm{Up})=2.0000
$$


$$
\gamma^{2}Q_t=(0.8100)(2.0000)=1.6200
$$


$$
\mathbf g_t=\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{2}Q_t\mathbf g_t\\&=\frac{1}{3}(1.6200)\begin{bmatrix}0.7500\\-0.2500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}0.4050\\-0.1350\\-0.1350\\-0.1350\end{bmatrix}\end{aligned}
$$


### Step t = 3: state S3, action Down

$$
Q_t=Q(S3,\mathrm{Down})=2.5000
$$


$$
\gamma^{3}Q_t=(0.7290)(2.5000)=1.8225
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{3}Q_t\mathbf g_t\\&=\frac{1}{3}(1.8225)\begin{bmatrix}-0.2500\\0.7500\\-0.2500\\-0.2500\end{bmatrix}\\&=\begin{bmatrix}-0.1519\\0.4556\\-0.1519\\-0.1519\end{bmatrix}\end{aligned}
$$


### Step t = 4: state S4, action Right

$$
Q_t=Q(S4,\mathrm{Right})=4.0000
$$


$$
\gamma^{4}Q_t=(0.6561)(4.0000)=2.6244
$$


$$
\mathbf g_t=\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}
$$


$$
\begin{aligned}\mathbf c_t&=\frac{1}{3}\gamma^{4}Q_t\mathbf g_t\\&=\frac{1}{3}(2.6244)\begin{bmatrix}-0.2500\\-0.2500\\-0.2500\\0.7500\end{bmatrix}\\&=\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}\end{aligned}
$$


## Average the three trajectories and update once

### State S0

$$
\hat{\mathbf g}_Q(S0)=\begin{bmatrix}-0.3333\\-0.3333\\-1.0000\\1.6667\end{bmatrix}
$$


$$
\mathbf z_{new}(S0)=0.1\begin{bmatrix}-0.3333\\-0.3333\\-1.0000\\1.6667\end{bmatrix}=\begin{bmatrix}-0.0333\\-0.0333\\-0.1000\\0.1667\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S0)=\begin{bmatrix}0.2406\\0.2406\\0.2250\\0.2938\end{bmatrix}
$$


### State S1

$$
\hat{\mathbf g}_Q(S1)=\begin{bmatrix}0.7500\\-0.1500\\-0.7500\\0.1500\end{bmatrix}
$$


$$
\mathbf z_{new}(S1)=0.1\begin{bmatrix}0.7500\\-0.1500\\-0.7500\\0.1500\end{bmatrix}=\begin{bmatrix}0.0750\\-0.0150\\-0.0750\\0.0150\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S1)=\begin{bmatrix}0.2691\\0.2459\\0.2316\\0.2534\end{bmatrix}
$$


### State S2

$$
\hat{\mathbf g}_Q(S2)=\begin{bmatrix}0.2700\\-0.5400\\-0.2700\\0.5400\end{bmatrix}
$$


$$
\mathbf z_{new}(S2)=0.1\begin{bmatrix}0.2700\\-0.5400\\-0.2700\\0.5400\end{bmatrix}=\begin{bmatrix}0.0270\\-0.0540\\-0.0270\\0.0540\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S2)=\begin{bmatrix}0.2566\\0.2366\\0.2431\\0.2636\end{bmatrix}
$$


### State S3

$$
\hat{\mathbf g}_Q(S3)=\begin{bmatrix}-0.2430\\0.3645\\-0.2430\\0.1215\end{bmatrix}
$$


$$
\mathbf z_{new}(S3)=0.1\begin{bmatrix}-0.2430\\0.3645\\-0.2430\\0.1215\end{bmatrix}=\begin{bmatrix}-0.0243\\0.0365\\-0.0243\\0.0122\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S3)=\begin{bmatrix}0.2439\\0.2592\\0.2439\\0.2530\end{bmatrix}
$$


### State S4

$$
\hat{\mathbf g}_Q(S4)=\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}
$$


$$
\mathbf z_{new}(S4)=0.1\begin{bmatrix}-0.2187\\-0.2187\\-0.2187\\0.6561\end{bmatrix}=\begin{bmatrix}-0.0219\\-0.0219\\-0.0219\\0.0656\end{bmatrix}
$$


$$
\boldsymbol{\pi}_{new}(S4)=\begin{bmatrix}0.2444\\0.2444\\0.2444\\0.2668\end{bmatrix}
$$


### How to read the Q-weighted Policy Gradient equation

The learning signal is the action value itself:

$$
Q(S_t,A_t).
$$

The batch gradient estimate is

$$
\hat{\mathbf g}_{Q}
=
\frac{1}{3}
\sum_{i=1}^{3}
\sum_{t=0}^{T_i-1}
\gamma^t
Q(S_t^{(i)},A_t^{(i)})
\nabla_{\theta}
\log\pi_{\theta}
\left(
A_t^{(i)}\mid S_t^{(i)}
\right).
$$

Meaning:

- $Q(S_t,A_t)$: critic's estimate of the expected future return for the selected state-action pair;
- inner $\sum_t$: combine all step contributions inside one trajectory;
- outer $\sum_i$: combine the three sampled trajectories;
- $1/3$: average those trajectory contributions.

**How to read it:** actions with larger positive $Q$-values receive stronger positive reinforcement; actions with negative $Q$-values push the policy in the opposite direction.

In [12]:

# Q-weighted PG verification
grad_q = {s:np.zeros(4) for s in states}

for traj in trajectories:
    for t,(s,a,r) in enumerate(traj):
        q = Q[s][a]
        grad_q[s] += ((gamma**t)*q*grad_log_softmax(p0,a))/N

q_logits = {s:alpha*grad_q[s] for s in states}
pd.DataFrame({s:softmax(q_logits[s]) for s in states}, index=ACTIONS).T


,Up,Down,Left,Right
S0,0.2406,0.2406,0.2250,0.2938
S1,0.2691,0.2459,0.2316,0.2534
S2,0.2566,0.2366,0.2431,0.2636
S3,0.2439,0.2592,0.2439,0.2530
S4,0.2444,0.2444,0.2444,0.2668



# 9. Final comparison

REINFORCE uses

$$
\gamma^tG_t.
$$

Vanilla PG uses the average of trajectory gradients based on

$$
\gamma^tG_t.
$$

Advantage PG uses

$$
\gamma^t\left[Q(S_t,A_t)-V(S_t)\right].
$$

TD-residual PG uses

$$
\gamma^t\left[R_{t+1}+\gamma V(S_{t+1})-V(S_t)\right].
$$

Q-weighted PG uses

$$
\gamma^tQ(S_t,A_t).
$$

All of them multiply a learning signal by

$$
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$


## 9.1 One equation that unifies all five methods

All the methods in this notebook can be read through

$$
\boxed{
\nabla_{\theta}J(\theta)
\approx
\text{average over sampled trajectories}
\left[
\text{sum over time steps}
\left(
\text{learning signal}
\times
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t)
\right)
\right].
}
$$

The policy-gradient part stays the same:

$$
\nabla_{\theta}\log\pi_{\theta}(A_t\mid S_t).
$$

Only the learning signal changes:

$$
\begin{array}{ll}
\text{REINFORCE / VPG:} & \gamma^tG_t,\\[4pt]
\text{Advantage PG:} & \gamma^t\left[Q(S_t,A_t)-V(S_t)\right],\\[4pt]
\text{TD-residual PG:} & \gamma^t\delta_t,\\[4pt]
\text{Q-weighted PG:} & \gamma^tQ(S_t,A_t).
\end{array}
$$

A compact way to read the whole family is:

> **Which sampled action should become more or less probable, and by how much?**

The score-function gradient identifies **which action probability to change**.  
The learning signal determines **the direction and strength of that change**.